In [1]:
import pandas as pd
import numpy as np

In [2]:
# cell 2
attendance = pd.read_excel("../data/Attendance_Juli.xlsx")

attendance_work = attendance[
    [
        "Date",
        "Employee No",
        "Employee Name",
        "Division",
        "Attendance Status",
        "Time In",
        "Time Out",
        "Late In"
    ]
].copy()

#count day
pattern = "Present|Business Trip|Present Off|Absent|Leave|Permit"

attendance_work["Count Day"] = (
    attendance_work["Attendance Status"]
    .fillna("")
    .str.contains(pattern, case=False, regex=True)
    .astype(int)
)

#clock In
attendance_work["Clock In"] = (
    attendance_work["Time In"].notna() |
    attendance_work["Attendance Status"]
    .fillna("")
    .str.contains("Leave", case=False)
).astype(int)

#clock Out
attendance_work["Clock Out"] = (
    attendance_work["Time Out"].notna() |
    attendance_work["Attendance Status"]
    .fillna("")
    .str.contains("Leave", case=False)
).astype(int)

#late   
attendance_work["Late"] = (
    attendance_work["Late In"] > 0
).astype(int)

# Leave (untuk kompensasi EDP)
attendance_work["Leave"] = (
    attendance_work["Attendance Status"]
    .fillna("")
    .str.strip()
    .eq("Leave")
).astype(int)

In [3]:
#cell 3
summary = (
    attendance_work
    .groupby(["Employee No", "Employee Name"])
    .agg(
        Date=("Date", "count"),
        Division=("Division", "first"),
        Count_Day=("Count Day", "sum"),
        Clock_In=("Clock In", "sum"),
        Clock_Out=("Clock Out", "sum"),
        Late_Count=("Late", "sum"),
        Leave_Count=("Leave", "sum")
    )
    .reset_index()
)

In [4]:
#cell 4
summary["C.In"] = (
    summary["Clock_In"] / summary["Count_Day"]
).clip(upper=1)

summary["C.Out"] = (
    summary["Clock_Out"] / summary["Count_Day"]
).clip(upper=1)

summary["Late"] = (
    summary["Late_Count"] / summary["Count_Day"]
).clip(upper=1)

summary["Leave_EDP"] = (
    summary["Leave_Count"] / summary["Count_Day"]
).clip(upper=1)

In [5]:
#cell 5
summary["C.In"] = summary["C.In"].map("{:.0%}".format)
summary["C.Out"] = summary["C.Out"].map("{:.0%}".format)
summary["Late"] = summary["Late"].map("{:.0%}".format)

# Leave_EDP jangan diformat dulu

In [6]:
#cell 6
display(attendance_work.head(20))

,Date,Employee No,Employee Name,Division,Attendance Status,Time In,Time Out,Late In,Count Day,Clock In,Clock Out,Late,Leave
0,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0
1,2026-07-01,E1301001,Ignatio Immanuel,"Finance, Accounting, Tax Division ...",Absent,NaN,NaN,0.0,1,0,0,0,0
2,2026-07-02,E1301001,Ignatio Immanuel,"Finance, Accounting, Tax Division ...",Absent,NaN,NaN,0.0,1,0,0,0,0
3,2026-07-03,E1301001,Ignatio Immanuel,"Finance, Accounting, Tax Division ...",Absent,NaN,NaN,0.0,1,0,0,0,0
4,2026-07-04,E1301001,Ignatio Immanuel,"Finance, Accounting, Tax Division ...",Off,NaN,NaN,0.0,0,0,0,0,0
5,2026-07-05,E1301001,Ignatio Immanuel,"Finance, Accounting, Tax Division ...",Off,NaN,NaN,0.0,0,0,0,0,0
6,2026-07-06,E1301001,Ignatio Immanuel,"Finance, Accounting, Tax Division ...",Absent,NaN,NaN,0.0,1,0,0,0,0
7,2026-07-07,E1301001,Ignatio Immanuel,"Finance, Accounting, Tax Division ...",Absent,NaN,NaN,0.0,1,0,0,0,0
8,2026-07-08,E1301001,Ignatio Immanuel,"Finance, Accounting, Tax Division ...",Absent,NaN,NaN,0.0,1,0,0,0,0
9,2026-07-09,E1301001,Ignatio Immanuel,"Finance, Accounting, Tax Division ...",Absent,NaN,NaN,0.0,1,0,0,0,0


In [7]:
#cell 7
summary = summary.sort_values("Employee Name")

In [8]:
#cell 8
display(summary)

,Employee No,Employee Name,Date,Division,Count_Day,Clock_In,Clock_Out,Late_Count,Leave_Count,C.In,C.Out,Late,Leave_EDP
755,X2002454,A. Arya Nugraha,31,Service Operation Division ...,23,23,18,18,0,100%,78%,78%,0.000000
233,E2101779,Abdi Pratama Putra,31,Network Operation Division ...,23,23,23,0,1,100%,100%,0%,0.043478
591,E25021397,Abdul Gani,31,Service Operation Division ...,23,22,7,21,0,96%,30%,91%,0.000000
845,X2204657,Abdul Khair,31,Service Operation Division ...,31,31,30,0,0,100%,97%,0%,0.000000
208,E2101664,Abdul Rasid,31,Customer Care & Service Assurance Management D...,23,20,14,17,0,87%,61%,74%,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
974,X2404905,Zhafir Ridho,31,Service Operation Division ...,24,23,22,22,0,96%,92%,92%,0.000000
903,X2304821,Ziar Ahriz Rizkiya,31,Service Operation Division ...,25,22,17,5,0,88%,68%,20%,0.000000
1048,X2502990,Ziyad Ulwan Hadi Nasution,31,Network Development Division ...,23,23,21,22,0,100%,91%,96%,0.000000
378,E23031072,Zulhan Dwi Ikhsani,31,Regional Sales Division ...,23,23,18,16,0,100%,78%,70%,0.000000


Perhitungan File EDP

In [9]:
# cell 9
edp = pd.read_excel("../data/EDP_Juli.xlsx")

print(edp.shape)
display(edp.head())

(16254, 33)


,Date,Employee Name,Company,Job Title,Organization,Work Location,Work Location Type,Work Location Area,Project Name,Task,...,SLA Target Minutes,SLA Actual Minutes,SLA Status,Productivity Target,Productivity Actual,Productivity Score,Hourly Rate Source,Hourly Rate,Total Point,Status Verification
0,2026-07-01,(E2201880) Sapto Handoyo Jati,FS,Government Sales Department Head,Government Sales Department,Head Quarter,HQ,HQ,Sales Division,00000787-Internal Meeting/Coordination,...,300.0,120.0,On Time,60.0,60.0,0.0,NaN,0.0,NaN,NaN
1,2026-07-01,(E2201880) Sapto Handoyo Jati,FS,Government Sales Department Head,Government Sales Department,Head Quarter,HQ,HQ,Sales Division,00000788-External Meeting,...,300.0,60.0,On Time,60.0,60.0,0.0,NaN,0.0,NaN,NaN
2,2026-07-01,(E1501021) Dikken Octora,FS,LMS Operation Delivery Engineer,LMS Operation Delivery Unit,Bandung,Central,West Java,Operation,00000587-Reconsiliation,...,300.0,255.0,On Time,60.0,60.0,0.0,NaN,0.0,NaN,NaN
3,2026-07-01,(E1501021) Dikken Octora,FS,LMS Operation Delivery Engineer,LMS Operation Delivery Unit,Bandung,Central,West Java,Operation,00000603-Data & Documentation,...,300.0,120.0,On Time,60.0,60.0,0.0,NaN,0.0,NaN,NaN
4,2026-07-01,(E1501021) Dikken Octora,FS,LMS Operation Delivery Engineer,LMS Operation Delivery Unit,Bandung,Central,West Java,Operation,00000597-Reports Daily,...,300.0,125.0,On Time,60.0,60.0,0.0,NaN,0.0,NaN,NaN


In [10]:
#cell 10
print(edp.columns.tolist())

# Ambil Employee No
edp["Employee No"] = (
    edp["Employee Name"]
    .str.extract(r"\((.*?)\)", expand=False)
    .str.strip()
)

# Bersihkan nama
edp["Employee Name"] = (
    edp["Employee Name"]
    .str.replace(r"\(.*?\)", "", regex=True)
    .str.strip()
)

# Format tanggal
edp["Date"] = pd.to_datetime(edp["Date"]).dt.normalize()

# Satu orang satu hari dihitung sekali
edp = edp.drop_duplicates(
    subset=["Employee No", "Date"]
)

display(edp.head())

['Date', 'Employee Name', 'Company', 'Job Title', 'Organization', 'Work Location', 'Work Location Type', 'Work Location Area', 'Project Name', 'Task', 'Task Period Type', 'Task Weight', 'Percent Complete', 'Source Type', 'Shift Name', 'Schedule - Time In', 'Schedule - Time Out', 'Schedule - Working Hours', 'Timesheet - Time In', 'Timesheet - Time Out', 'Timesheet - Working Hours', 'Percent Working', 'Billable', 'SLA Target Minutes', 'SLA Actual Minutes', 'SLA Status', 'Productivity Target', 'Productivity Actual', 'Productivity Score', 'Hourly Rate Source', 'Hourly Rate', 'Total Point', 'Status Verification']


,Date,Employee Name,Company,Job Title,Organization,Work Location,Work Location Type,Work Location Area,Project Name,Task,...,SLA Actual Minutes,SLA Status,Productivity Target,Productivity Actual,Productivity Score,Hourly Rate Source,Hourly Rate,Total Point,Status Verification,Employee No
0,2026-07-01,Sapto Handoyo Jati,FS,Government Sales Department Head,Government Sales Department,Head Quarter,HQ,HQ,Sales Division,00000787-Internal Meeting/Coordination,...,120.0,On Time,60.0,60.0,0.0,NaN,0.0,NaN,NaN,E2201880
2,2026-07-01,Dikken Octora,FS,LMS Operation Delivery Engineer,LMS Operation Delivery Unit,Bandung,Central,West Java,Operation,00000587-Reconsiliation,...,255.0,On Time,60.0,60.0,0.0,NaN,0.0,NaN,NaN,E1501021
5,2026-07-01,Su Hua,FS,Billing & Collection Department Head,Billing & Collection Department,Head Quarter,HQ,HQ,Finance,00000161-Billing & Collection,...,30.0,On Time,60.0,30.0,0.0,NaN,0.0,NaN,NaN,E2001564
10,2026-07-01,Mochammad Syarif Hidayaturrochman,FS,Account Manager,Branch Central Java Department,Semarang,Central,Central Java,Regional,00000783-Administration,...,61.0,On Time,60.0,61.0,0.0,NaN,0.0,NaN,NaN,E2203866
11,2026-07-01,Vernande Nirohito,FS,Project Administration & Monitoring Unit Head,Project Administration & Monitoring Unit,Head Quarter,HQ,HQ,NetDev,00000552-Report Project,...,60.0,On Time,60.0,60.0,0.0,NaN,0.0,NaN,NaN,E1701285


In [11]:
#cell 11
edp_summary = (
    edp
    .groupby(
        ["Employee No","Employee Name"],
        as_index=False
    )
    .agg(
        EDP_Count=("Date","nunique")
    )
)

display(edp_summary)

,Employee No,Employee Name,EDP_Count
0,E1401002,Fenny Ferdian,1
1,E1404008,Budirahayu,5
2,E1404013,Ade Yayan,13
3,E1501021,Dikken Octora,11
4,E1501051,Okta Reza Fiandy,3
...,...,...,...
450,X26011132,Yehezkiel Hartono,21
451,X26011134,Arief Sudarmawan,3
452,X26011135,Indah Sari Aritonang,23
453,X26011137,Denis Mahardhika,2


In [12]:
#cell 12
edp_only = (
    summary[
        [
            "Employee No",
            "Employee Name",
            "Count_Day"
        ]
    ]
    .merge(
        edp_summary,
        on=[
            "Employee No",
            "Employee Name"
        ],
        how="left"
    )
)

edp_only["EDP_Count"] = (
    edp_only["EDP_Count"]
    .fillna(0)
)

edp_only["EDP"] = (
    edp_only["EDP_Count"] /
    edp_only["Count_Day"]
)

edp_only["EDP"] = (
    edp_only["EDP"]
    .map("{:.2%}".format)
)

display(edp_only)

,Employee No,Employee Name,Count_Day,EDP_Count,EDP
0,X2002454,A. Arya Nugraha,23,0.0,0.00%
1,E2101779,Abdi Pratama Putra,23,22.0,95.65%
2,E25021397,Abdul Gani,23,0.0,0.00%
3,X2204657,Abdul Khair,31,23.0,74.19%
4,E2101664,Abdul Rasid,23,0.0,0.00%
...,...,...,...,...,...
1102,X2404905,Zhafir Ridho,24,0.0,0.00%
1103,X2304821,Ziar Ahriz Rizkiya,25,0.0,0.00%
1104,X2502990,Ziyad Ulwan Hadi Nasution,23,21.0,91.30%
1105,E23031072,Zulhan Dwi Ikhsani,23,23.0,100.00%


In [13]:
#cell 13
summary = summary.merge(
    edp_summary[
        [
            "Employee No",
            "Employee Name",
            "EDP_Count"
        ]
    ],
    on=[
        "Employee No",
        "Employee Name"
    ],
    how="left"
)

summary["EDP_Count"] = (
    summary["EDP_Count"]
    .fillna(0)
)

summary["EDP_Total"] = (
    summary["EDP_Count"] +
    summary["Leave_Count"]
)

summary["EDP_Total"] = np.minimum(
    summary["EDP_Total"],
    summary["Count_Day"]
)

summary["EDP"] = (
    summary["EDP_Total"] /
    summary["Count_Day"]
)

summary["EDP"] = summary["EDP"].map("{:.0%}".format)

In [14]:
summary_final = (
    summary[
        [
            "Employee No",
            "Employee Name",
            "Division",
            "Count_Day",
            "Leave_Count",
            "EDP_Count",
            "EDP_Total",
            "C.In",
            "C.Out",
            "Late",
            "EDP"
        ]
    ]
    .sort_values("Employee Name")
)

display(summary_final)

,Employee No,Employee Name,Division,Count_Day,Leave_Count,EDP_Count,EDP_Total,C.In,C.Out,Late,EDP
0,X2002454,A. Arya Nugraha,Service Operation Division ...,23,0,0.0,0.0,100%,78%,78%,0%
1,E2101779,Abdi Pratama Putra,Network Operation Division ...,23,1,22.0,23.0,100%,100%,0%,100%
2,E25021397,Abdul Gani,Service Operation Division ...,23,0,0.0,0.0,96%,30%,91%,0%
3,X2204657,Abdul Khair,Service Operation Division ...,31,0,23.0,23.0,100%,97%,0%,74%
4,E2101664,Abdul Rasid,Customer Care & Service Assurance Management D...,23,0,0.0,0.0,87%,61%,74%,0%
...,...,...,...,...,...,...,...,...,...,...,...
1102,X2404905,Zhafir Ridho,Service Operation Division ...,24,0,0.0,0.0,96%,92%,92%,0%
1103,X2304821,Ziar Ahriz Rizkiya,Service Operation Division ...,25,0,0.0,0.0,88%,68%,20%,0%
1104,X2502990,Ziyad Ulwan Hadi Nasution,Network Development Division ...,23,0,21.0,21.0,100%,91%,96%,91%
1105,E23031072,Zulhan Dwi Ikhsani,Regional Sales Division ...,23,0,23.0,23.0,100%,78%,70%,100%


In [15]:
output_file = "../output/Summary_Juli_2026.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    for division, data in summary_final.groupby("Division"):

        # nama sheet maksimal 31 karakter
        sheet_name = division[:31]

        data = data.sort_values("Employee Name")

        data.to_excel(
            writer,
            sheet_name=sheet_name,
            index=False
        )

print("Selesai")

Selesai
